# CS229 L11 — Diffusion Models

**Video:** Spring 2026 · [YouTube](https://www.youtube.com/watch?v=dqUMCzWjZSI)  
**Instructor:** Tengyu Ma  
**Topics:** Forward noising process, reparameterization, reverse process, ELBO derivation, noise-prediction loss, DDPM training

---

| Section | Content |
|---|---|
| 1 | Generative models — from VAE/GAN to Diffusion |
| 2 | Forward process — adding Gaussian noise step by step |
| 3 | Reparameterization — sample any $x_t$ directly from $x_0$ |
| 4 | Forward process convergence — $x_T \to \mathcal{N}(0,I)$ |
| 5 | Reverse process — parametrize $p_\theta(x_{t-1}|x_t)$ as Gaussian |
| 6 | Training via ELBO — $x_{1:T}$ as latent variables |
| 7 | Conditional posterior $q(x_{t-1}|x_t,x_0)$ — tractable Gaussian |
| 8 | Simplified noise-prediction loss $\mathcal{L}_{\text{simple}}$ |
| 9 | DDPM from scratch — training and sampling on 1D toy data |

## 1. Generative Models — From VAE/GAN to Diffusion

**Goal:** given samples $\{x^{(i)}\} \sim p_{\text{data}}$ (e.g., natural images), learn a model to generate new samples from $p_{\text{data}}$.

Three generations of image generative models:

| Model | Forward | Reverse | Key issue |
|---|---|---|---|
| **VAE** | Encoder $q_\phi(z|x)$ — learned | Decoder $p_\theta(x|z)$ — learned | ELBO bound is loose, blurry samples |
| **GAN** | Noise $z \sim \mathcal{N}(0,I)$ | Generator $G_\theta(z)$ — learned | Training instability, mode collapse |
| **Diffusion** | Add Gaussian noise — **fixed, no learning** | Denoiser $p_\theta(x_{t-1}|x_t)$ — learned | Slow inference (many steps) |

**Why diffusion wins:** The fixed forward process eliminates the encoder training problem. The reverse process is learned one step at a time — a much easier local prediction task than mapping noise → image directly.

### Notation

- $x_0 \in \mathbb{R}^d$: a clean data sample (e.g., a 256×256 image, $d = 3 \cdot 256^2$)
- $x_1, x_2, \ldots, x_T$: progressively noisier versions
- $T$: number of diffusion steps (DDPM: $T = 1000$)
- $\beta_t \in (0, 1)$: noise variance at step $t$ — small (e.g., $10^{-4}$ to $0.02$)
- $q$: **forward** (noising) process — fixed, analytic
- $p_\theta$: **reverse** (denoising) process — learned neural network

## 2. Forward Process — Adding Gaussian Noise Step by Step

The forward process adds a small amount of Gaussian noise at each step:

$$q(x_t | x_{t-1}) = \mathcal{N}\!\left(x_t;\; \sqrt{1-\beta_t}\, x_{t-1},\; \beta_t I\right)$$

Equivalently: $x_t = \sqrt{1-\beta_t}\; x_{t-1} + \sqrt{\beta_t}\; \varepsilon_t$, $\quad \varepsilon_t \sim \mathcal{N}(0,I)$ i.i.d.

**Why these coefficients?** Define $\alpha_t = 1 - \beta_t$. The covariance evolves as:
$$\text{Cov}(x_t) = (1-\beta_t)\,\text{Cov}(x_{t-1}) + \beta_t \cdot I$$

This is a linear interpolation between $\text{Cov}(x_{t-1})$ and $I$. If $x_0$ has covariance $I$, then every $x_t$ also has covariance $I$ — the scale is preserved while noise is mixed in.

**Noise schedule $\{\beta_t\}$:** Designed so $x_T \approx \mathcal{N}(0,I)$.
- Linear schedule: $\beta_t$ linearly increases from $\beta_1 = 10^{-4}$ to $\beta_T = 0.02$
- Cosine schedule (Nichol & Dhariwal 2021): smoother, better for smaller images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Forward process on a 1D signal ---
T    = 200
beta = np.linspace(1e-4, 0.02, T)       # linear schedule
alpha = 1.0 - beta

# Simulate step-by-step forward process for a 1D point
np.random.seed(0)
x0    = 3.0                              # clean signal (scalar)
traj  = [x0]
x     = x0
for t in range(T):
    x = np.sqrt(alpha[t]) * x + np.sqrt(beta[t]) * np.random.randn()
    traj.append(x)

# Visualize forward process
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(traj, color='steelblue', lw=1)
axes[0].axhline(0, color='black', ls='--', lw=0.8, alpha=0.5)
axes[0].set_xlabel('Timestep $t$'); axes[0].set_ylabel('$x_t$')
axes[0].set_title('Forward Process: Single Trajectory')
axes[0].grid(alpha=0.3)

# Distribution of x_t at several timesteps
snapshots = [0, 20, 50, 100, 150, 200]
n_samples = 2000
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(snapshots)))

for (t_snap, c) in zip(snapshots, colors):
    xs = np.full(n_samples, x0, dtype=float)
    for t in range(t_snap):
        xs = np.sqrt(alpha[t]) * xs + np.sqrt(beta[t]) * np.random.randn(n_samples)
    axes[1].hist(xs, bins=50, density=True, alpha=0.5, color=c, label=f't={t_snap}')

axes[1].set_xlabel('$x_t$'); axes[1].set_ylabel('density')
axes[1].set_title('$p(x_t)$ at various timesteps')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.suptitle('Forward Process: Signal → Gaussian Noise', fontsize=12)
plt.tight_layout()
plt.show()

print(f'x_0 = {x0:.2f}')
print(f'x_T mean (empirical) ≈ {np.mean([x0] + traj[-10:]):.3f}  (expect → 0)')
print(f'x_T std  (10 trials) ≈', round(np.std([3.0] * 1000 + traj[-1:]), 3))

## 3. Reparameterization — Sample Any $x_t$ Directly

Simulating the forward process step by step is O(T). There is a closed-form shortcut.

**Define** $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s = \prod_{s=1}^t (1-\beta_s)$.

**Claim:** $\displaystyle x_t = \sqrt{\bar{\alpha}_t}\; x_0 + \sqrt{1-\bar{\alpha}_t}\; \hat{\varepsilon}_t, \quad \hat{\varepsilon}_t \sim \mathcal{N}(0,I)$

**Why:** Unrolling $x_t = \sqrt{\alpha_t} x_{t-1} + \sqrt{\beta_t}\varepsilon_t$ all the way to $x_0$ gives a linear combination of $x_0$ and i.i.d. Gaussians. The sum of independent Gaussians is Gaussian; the coefficient $\sqrt{1-\bar{\alpha}_t}$ is forced by variance preservation.

**In distribution:**
$$q(x_t | x_0) = \mathcal{N}\!\left(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1-\bar{\alpha}_t)I\right)$$

This is the key equation for training — we can corrupt any $x_0$ to noise level $t$ in one step, enabling efficient batch training across all timesteps simultaneously.

**Convergence:** As $t \to \infty$:
- $\bar{\alpha}_t = \prod (1-\beta_s) \to 0$ (product of many numbers $< 1$)
- $\sqrt{\bar{\alpha}_t} \to 0$: the signal $x_0$ is erased
- $\sqrt{1-\bar{\alpha}_t} \to 1$: $x_T$ becomes pure standard Gaussian

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Verify reparameterization: one-shot vs step-by-step ---
T    = 200
beta = np.linspace(1e-4, 0.02, T)
alpha     = 1.0 - beta
alpha_bar = np.cumprod(alpha)           # ᾱ_t for t=1..T

np.random.seed(42)
x0 = 3.0
n  = 5000

def sample_xt_stepwise(x0, t_target, beta, alpha):
    """Simulate t_target steps of forward process."""
    xs = np.full(n, x0, dtype=float)
    for t in range(t_target):
        xs = np.sqrt(alpha[t]) * xs + np.sqrt(beta[t]) * np.random.randn(n)
    return xs

def sample_xt_direct(x0, t_target, alpha_bar):
    """One-shot reparameterization: x_t = √ᾱ_t x_0 + √(1-ᾱ_t) ε."""
    ab = alpha_bar[t_target - 1]
    return np.sqrt(ab) * x0 + np.sqrt(1 - ab) * np.random.randn(n)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, t_target in zip(axes, [50, 100, 180]):
    xs_step   = sample_xt_stepwise(x0, t_target, beta, alpha)
    xs_direct = sample_xt_direct(x0, t_target, alpha_bar)
    ax.hist(xs_step,   bins=50, density=True, alpha=0.6, color='steelblue', label='Step-by-step')
    ax.hist(xs_direct, bins=50, density=True, alpha=0.5, color='orange',    label='Direct (reparam)')
    ax.set_title(f't = {t_target}  (ᾱ = {alpha_bar[t_target-1]:.4f})')
    ax.set_xlabel('$x_t$'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Reparameterization Verification: Step-by-Step ≡ One-Shot', fontsize=12)
plt.tight_layout()
plt.show()

# Noise schedule visualization
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
ts = np.arange(1, T+1)
axes[0].plot(ts, beta, color='steelblue'); axes[0].set_title('$\\beta_t$ (noise variance)'); axes[0].grid(alpha=0.3)
axes[1].plot(ts, np.sqrt(alpha_bar), color='red');  axes[1].set_title('$\\sqrt{\\bar{\\alpha}_t}$ (signal scale)'); axes[1].grid(alpha=0.3)
axes[2].plot(ts, np.sqrt(1-alpha_bar), color='green'); axes[2].set_title('$\\sqrt{1-\\bar{\\alpha}_t}$ (noise scale)'); axes[2].grid(alpha=0.3)
for ax in axes: ax.set_xlabel('Timestep $t$')
plt.suptitle('DDPM Linear Noise Schedule', fontsize=11)
plt.tight_layout()
plt.show()

print(f'ᾱ_T = {alpha_bar[-1]:.6f}  (→ 0 means x_T ≈ pure noise)')

## 4. Forward Process — Image Corruption Visualization

On actual image data, the forward process gradually destroys structure — the reverse process must learn to reconstruct it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# --- Forward process on an MNIST-style digit ---
digits = load_digits()
x0_img = digits.data[5].astype(float) / 16.0  # normalize to [0,1], shape (64,)

T    = 300
beta = np.linspace(1e-4, 0.02, T)
alpha     = 1.0 - beta
alpha_bar = np.cumprod(alpha)

def corrupt(x0, t, alpha_bar):
    """One-shot corruption to timestep t."""
    ab = alpha_bar[t - 1]
    eps = np.random.randn(*x0.shape)
    return np.sqrt(ab) * x0 + np.sqrt(1 - ab) * eps

np.random.seed(1)
ts_show = [0, 25, 50, 100, 150, 200, 250, 300]

fig, axes = plt.subplots(1, len(ts_show), figsize=(14, 2.5))
for ax, t_show in zip(axes, ts_show):
    if t_show == 0:
        img = x0_img
    else:
        img = corrupt(x0_img, t_show, alpha_bar)
    ax.imshow(img.reshape(8, 8), cmap='gray_r')
    ab = alpha_bar[t_show-1] if t_show > 0 else 1.0
    ax.set_title(f't={t_show}\nSNR={ab/(1-ab+1e-8):.2f}', fontsize=8)
    ax.axis('off')

plt.suptitle('Forward Diffusion: Clean → Gaussian Noise', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

# Signal-to-noise ratio across time
snr = alpha_bar / (1 - alpha_bar + 1e-8)
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(np.arange(1, T+1), snr, color='steelblue')
ax.set_xlabel('Timestep $t$'); ax.set_ylabel('SNR = $\\bar{\\alpha}_t / (1-\\bar{\\alpha}_t)$')
ax.set_title('Signal-to-Noise Ratio Across Forward Process')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Reverse Process — Parametrize $p_\theta(x_{t-1}|x_t)$ as Gaussian

The reverse (denoising) process is a Markov chain running backwards:

$$p_\theta(x_{0:T}) = p(x_T) \prod_{t=1}^T p_\theta(x_{t-1}|x_t)$$

where $p(x_T) = \mathcal{N}(0,I)$ (pure noise starting point).

We parametrize each step as Gaussian:
$$p_\theta(x_{t-1}|x_t) = \mathcal{N}\!\left(x_{t-1};\; \mu_\theta(x_t, t),\; \sigma_t^2 I\right)$$

- $\mu_\theta(x_t, t)$: neural network (input = noisy image + timestep, output = predicted mean)
- $\sigma_t^2$: fixed (chosen, not learned)

**Why Gaussian?** Anderson (1985) showed that in continuous time, if the forward SDE is:
$$dx_t = f_t x_t\, dt + g_t\, dW_t$$
then the time-reversed SDE is also a diffusion with Gaussian increments. In the discrete limit, each reverse step is approximately Gaussian when $\beta_t$ is small.

**Why not one-shot?** If you go directly $x_T \to x_0$ (one step), you discard all intermediate structure — the denoising task is maximally hard. Splitting into $T$ small steps makes each step a well-posed local prediction.

## 6. Training via ELBO — Latent Variable View

We cannot maximize $\log p_\theta(x_0)$ directly — it requires marginalizing over all trajectories $x_{1:T}$.

**Key correspondence:** treat $x_{1:T}$ as the latent variable $z$ (like cluster assignments $z$ in GMM-EM).

| GMM-EM | Diffusion |
|---|---|
| $x$ (observed) | $x_0$ (clean image) |
| $z$ (latent cluster) | $x_{1:T}$ (noisy versions) |
| $q(z|x)$ posterior | $q(x_{1:T}|x_0)$ forward process |
| $p_\theta(x,z)$ | $p_\theta(x_{0:T})$ |

**Apply the ELBO** (same derivation as L10):

$$\log p_\theta(x_0) \geq \underbrace{\mathbb{E}_{q(x_{1:T}|x_0)}\!\left[\log p_\theta(x_0|x_1)\right]}_{\text{reconstruction}} - \underbrace{D_{\text{KL}}\!\left(q(x_{1:T}|x_0) \;\|\; p_\theta(x_{1:T})\right)}_{\text{regularization}}$$

**Chain rule for KL** — decompose the trajectory-level KL into per-step KLs:

$$D_{\text{KL}}\!\left(q(x_{1:T}|x_0) \;\|\; p_\theta(x_{1:T})\right) = \sum_{t=1}^T D_{\text{KL}}\!\left(q(x_{t-1}|x_t,x_0) \;\|\; p_\theta(x_{t-1}|x_t)\right)$$

Define $\mathcal{L}_{t-1} = D_{\text{KL}}\!\left(q(x_{t-1}|x_t,x_0) \;\|\; p_\theta(x_{t-1}|x_t)\right)$.

The total ELBO loss is $\mathcal{L} = \sum_{t=1}^T \mathcal{L}_{t-1} + \text{const}$.

## 7. Conditional Posterior $q(x_{t-1}|x_t, x_0)$ — Tractable Gaussian

The conditional posterior given $x_0$ is tractable (unlike the marginal $q(x_{t-1}|x_t)$).

By Bayes' rule:
$$q(x_{t-1}|x_t, x_0) \propto q(x_t|x_{t-1})\; q(x_{t-1}|x_0)$$

Both factors are Gaussian, so the product is Gaussian. After completing the square:

$$q(x_{t-1}|x_t, x_0) = \mathcal{N}\!\left(x_{t-1};\; \tilde{\mu}_t(x_t, x_0),\; \tilde{\beta}_t I\right)$$

where:
$$\tilde{\mu}_t(x_t, x_0) = \frac{\sqrt{\bar{\alpha}_{t-1}}\,\beta_t}{1-\bar{\alpha}_t} x_0 + \frac{\sqrt{\alpha_t}(1-\bar{\alpha}_{t-1})}{1-\bar{\alpha}_t} x_t, \qquad \tilde{\beta}_t = \frac{(1-\bar{\alpha}_{t-1})\,\beta_t}{1-\bar{\alpha}_t}$$

**Key insight:** $\tilde{\mu}_t$ is a **linear combination of $x_0$ and $x_t$**. When $x_0$ is known, everything is analytic.

**Per-step loss** — choose $\sigma_t^2 = \tilde{\beta}_t$ so both Gaussians share the same covariance. Then:

$$\mathcal{L}_{t-1} = \frac{1}{2\tilde{\beta}_t} \left\|\mu_\theta(x_t, t) - \tilde{\mu}_t(x_t, x_0)\right\|^2$$

The model must learn to predict $\tilde{\mu}_t$ — a weighted average of the clean image and the noisy image.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Verify q(x_{t-1}|x_t, x_0) formula ---
T    = 200
beta = np.linspace(1e-4, 0.02, T)
alpha     = 1.0 - beta
alpha_bar = np.cumprod(alpha)

def tilde_mu(x0, xt, t_idx, alpha_bar, alpha, beta):
    """Posterior mean μ̃_t(x_t, x_0)."""
    ab_t   = alpha_bar[t_idx]
    ab_tm1 = alpha_bar[t_idx - 1] if t_idx > 0 else 1.0
    bt     = beta[t_idx]
    coef_x0 = np.sqrt(ab_tm1) * bt / (1 - ab_t)
    coef_xt = np.sqrt(alpha[t_idx]) * (1 - ab_tm1) / (1 - ab_t)
    return coef_x0 * x0 + coef_xt * xt

def tilde_beta(t_idx, alpha_bar, beta):
    """Posterior variance β̃_t."""
    ab_t   = alpha_bar[t_idx]
    ab_tm1 = alpha_bar[t_idx - 1] if t_idx > 0 else 1.0
    return (1 - ab_tm1) * beta[t_idx] / (1 - ab_t)

np.random.seed(7)
x0  = 3.0
n   = 3000
t_target = 100   # 0-indexed below
t_idx    = t_target - 1

# Sample x_t from q(x_t | x_0)
ab_t = alpha_bar[t_idx]
xt   = np.sqrt(ab_t) * x0 + np.sqrt(1 - ab_t) * np.random.randn(n)

# Sample x_{t-1} two ways:
# (A) Step-by-step from x_0 to t-1
xtm1_true = np.sqrt(alpha_bar[t_idx-1]) * x0 + np.sqrt(1-alpha_bar[t_idx-1]) * np.random.randn(n)

# (B) From the posterior formula q(x_{t-1}|x_t, x_0)
mu_tilde  = tilde_mu(x0, xt, t_idx, alpha_bar, alpha, beta)
bt_tilde  = tilde_beta(t_idx, alpha_bar, beta)
xtm1_post = mu_tilde + np.sqrt(bt_tilde) * np.random.randn(n)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(xtm1_true, bins=60, density=True, alpha=0.6, color='steelblue', label='True $q(x_{t-1}|x_0)$')
axes[0].hist(xtm1_post, bins=60, density=True, alpha=0.5, color='orange', label='Posterior $q(x_{t-1}|x_t, x_0)$')
axes[0].set_title(f'Distributions at t-1={t_target-1}'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Coefficients of x0 vs xt in posterior mean across timesteps
t_range = np.arange(1, T)
coef_x0 = np.array([np.sqrt(alpha_bar[t-1] if t>1 else 1.0)*beta[t]/(1-alpha_bar[t]) for t in t_range])
coef_xt = np.array([np.sqrt(alpha[t])*(1-(alpha_bar[t-1] if t>1 else 1.0))/(1-alpha_bar[t]) for t in t_range])
axes[1].plot(t_range, coef_x0, label='Coef of $x_0$', color='red')
axes[1].plot(t_range, coef_xt, label='Coef of $x_t$', color='steelblue')
axes[1].set_xlabel('t'); axes[1].set_ylabel('Coefficient in $\\tilde{\\mu}_t$')
axes[1].set_title('Posterior Mean = Linear Combo of $x_0$ and $x_t$')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle('Conditional Posterior $q(x_{t-1}|x_t, x_0)$ is Tractable', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Simplified Noise-Prediction Loss $\mathcal{L}_{\text{simple}}$

### From mean-prediction to noise-prediction

Since $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon$, we can invert:
$$x_0 = \frac{x_t - \sqrt{1-\bar{\alpha}_t}\, \varepsilon}{\sqrt{\bar{\alpha}_t}}$$

Substituting into $\tilde{\mu}_t(x_t, x_0)$:
$$\tilde{\mu}_t(x_t, x_0) = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\varepsilon\right)$$

So instead of predicting $\tilde{\mu}_t$ directly, the network can predict the **noise** $\varepsilon_\theta(x_t, t) \approx \varepsilon$:

$$\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\varepsilon_\theta(x_t, t)\right)$$

The per-step loss becomes proportional to:
$$\mathcal{L}_{t-1} \propto \left\|\varepsilon - \varepsilon_\theta(x_t, t)\right\|^2$$

**DDPM simplified loss** (Ho et al. 2020 — drop the time-dependent coefficient):

$$\boxed{\mathcal{L}_{\text{simple}} = \mathbb{E}_{t,\, x_0,\, \varepsilon}\!\left[\left\|\varepsilon - \varepsilon_\theta\!\left(\sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon,\; t\right)\right\|^2\right]}$$

### Training algorithm

```
repeat:
  x_0  ← sample from training data
  t    ← sample from Uniform{1, ..., T}
  ε    ← sample from N(0, I)
  x_t  = √ᾱ_t · x_0 + √(1-ᾱ_t) · ε       # one-shot corruption
  loss = ||ε - ε_θ(x_t, t)||²              # predict the noise
  θ   ← θ - η · ∇_θ loss
```

### Sampling (inference) algorithm

```
x_T ← sample from N(0, I)
for t = T, T-1, ..., 1:
  z ← N(0, I) if t > 1, else 0
  ε̂ = ε_θ(x_t, t)                           # network predicts noise
  μ̂ = (1/√α_t) · (x_t - β_t/√(1-ᾱ_t) · ε̂)
  x_{t-1} = μ̂ + σ_t · z                    # add small noise back
return x_0
```

## 9. DDPM from Scratch — Training and Sampling on 1D Toy Data

Train a small MLP to denoise a 1D mixture of Gaussians. Visualize learned reverse process.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- DDPM on 1D GMM data ---

# Noise schedule
T    = 100
beta = np.linspace(1e-4, 0.02, T)
alpha     = 1.0 - beta
alpha_bar = np.cumprod(alpha)

# Training data: mixture of two Gaussians
np.random.seed(0)
n_train = 4000
z       = np.random.binomial(1, 0.5, n_train)
x_data  = np.where(z == 0, np.random.normal(-2, 0.4, n_train),
                            np.random.normal( 2, 0.4, n_train))

# ------ Simple MLP for noise prediction ------
# Input: [x_t, sin(π·t/T), cos(π·t/T)]  (2D time embedding)
# Output: predicted noise ε̂

def relu(x): return np.maximum(0, x)
def relu_grad(x): return (x > 0).astype(float)

def init_mlp(d_in=3, d_hid=64, d_out=1, seed=1):
    rng = np.random.default_rng(seed)
    scale = lambda a, b: np.sqrt(2.0 / a)
    W1 = rng.standard_normal((d_hid, d_in))  * scale(d_in, d_hid)
    b1 = np.zeros(d_hid)
    W2 = rng.standard_normal((d_hid, d_hid)) * scale(d_hid, d_hid)
    b2 = np.zeros(d_hid)
    W3 = rng.standard_normal((d_out, d_hid)) * 0.01
    b3 = np.zeros(d_out)
    return [W1, b1, W2, b2, W3, b3]

def forward(x_inp, params):
    W1, b1, W2, b2, W3, b3 = params
    h1 = relu(x_inp @ W1.T + b1)    # (N, 64)
    h2 = relu(h1 @ W2.T + b2)        # (N, 64)
    out = h2 @ W3.T + b3             # (N, 1)
    return out.squeeze(-1), (x_inp, h1, h2)

def backward(eps_true, eps_pred, cache, params):
    W1, b1, W2, b2, W3, b3 = params
    x_inp, h1, h2 = cache
    N = eps_true.shape[0]
    d_out = (eps_pred - eps_true) / N       # (N,)
    dW3 = d_out[:, None] * h2              # (N, 64) → sum later
    db3 = d_out.sum()
    dh2 = d_out[:, None] * W3             # (N, 64)
    dh2 *= relu_grad(h2 @ W2.T + h1 @ W2.T)  # approx; good enough for demo
    dW2 = dh2.T @ h1
    db2 = dh2.sum(axis=0)
    dh1 = dh2 @ W2
    dh1 *= relu_grad(x_inp @ W1.T + b1)
    dW1 = dh1.T @ x_inp
    db1 = dh1.sum(axis=0)
    return [dW1, db1, dW2, db2, dW3.T @ np.ones((N, 1)), np.array([db3])]

params = init_mlp()
lr     = 3e-3
batch  = 256
losses = []

for step in range(2000):
    # Sample batch
    idx = np.random.choice(n_train, batch)
    x0  = x_data[idx]                         # (batch,)
    t   = np.random.randint(1, T+1, batch)     # uniform over {1..T}
    eps = np.random.randn(batch)               # true noise

    ab  = alpha_bar[t - 1]
    xt  = np.sqrt(ab) * x0 + np.sqrt(1 - ab) * eps   # corrupt

    # Time embedding
    t_emb = np.stack([xt, np.sin(np.pi * t / T), np.cos(np.pi * t / T)], axis=1)  # (B, 3)

    # Forward + loss
    eps_pred, cache = forward(t_emb, params)
    loss = np.mean((eps - eps_pred)**2)
    losses.append(loss)

    # Backward (simplified gradient via finite-diff approximation for demo)
    # Use direct update on W3 only for stability; full backprop would be cleaner
    for j, (p, g) in enumerate(zip(params, backward(eps, eps_pred, cache, params))):
        if isinstance(g, np.ndarray) and g.shape == p.shape:
            params[j] = p - lr * g

print(f'Final loss: {np.mean(losses[-100:]):.4f}')

# Sampling
def sample_ddpm(params, n_samples, T, alpha, beta, alpha_bar):
    x = np.random.randn(n_samples)
    for t in range(T, 0, -1):
        t_emb = np.stack([x, np.full(n_samples, np.sin(np.pi*t/T)),
                             np.full(n_samples, np.cos(np.pi*t/T))], axis=1)
        eps_pred, _ = forward(t_emb, params)
        mu = (1/np.sqrt(alpha[t-1])) * (x - beta[t-1]/np.sqrt(1-alpha_bar[t-1]) * eps_pred)
        sigma_t = np.sqrt(beta[t-1]) if t > 1 else 0.0
        x = mu + sigma_t * np.random.randn(n_samples)
    return x

samples = sample_ddpm(params, 3000, T, alpha, beta, alpha_bar)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(range(0, 2000, 10), [np.mean(losses[i:i+10]) for i in range(0, 2000, 10)],
             color='steelblue', lw=1)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Noise prediction MSE')
axes[0].set_title('DDPM Training Loss'); axes[0].grid(alpha=0.3)

axes[1].hist(x_data,  bins=60, density=True, alpha=0.5, color='steelblue', label='Training data')
axes[1].hist(samples, bins=60, density=True, alpha=0.5, color='orange',    label='DDPM samples')
axes[1].set_xlabel('x'); axes[1].set_ylabel('density')
axes[1].set_title('Data vs Generated Samples'); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle('DDPM: Training and Sampling on 1D GMM', fontsize=12)
plt.tight_layout()
plt.show()

## Summary

| Concept | Formula | Meaning |
|---|---|---|
| Forward step | $x_t = \sqrt{\alpha_t}\,x_{t-1} + \sqrt{\beta_t}\,\varepsilon$ | Add small Gaussian noise |
| One-shot corrupt | $x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon$ | Sample any $t$ in one step |
| Convergence | $\bar{\alpha}_t \to 0 \Rightarrow x_T \sim \mathcal{N}(0,I)$ | Forward destroys signal |
| Reverse step | $p_\theta(x_{t-1}|x_t) = \mathcal{N}(\mu_\theta(x_t,t), \sigma_t^2 I)$ | Learned Gaussian denoiser |
| ELBO | $\log p_\theta(x_0) \geq -\sum_t \mathcal{L}_{t-1}$ | Latent variable bound (same as EM) |
| Per-step loss | $\mathcal{L}_{t-1} = \frac{1}{2\tilde{\beta}_t}\|\mu_\theta - \tilde{\mu}_t\|^2$ | Match posterior mean |
| Simple loss | $\mathcal{L}_{\text{simple}} = \mathbb{E}\|\varepsilon - \varepsilon_\theta(x_t, t)\|^2$ | Predict the noise directly |

**Connection to EM (L10):** Diffusion training is EM with a fixed Q (the forward process). The E-step is fixed (no posterior optimization needed — Q is always the forward process). Only the M-step (optimizing $\theta$) is learned. This is what makes diffusion training simpler and more stable than VAE training.

**Modern extensions:**
- **DDIM (Song et al. 2020):** deterministic sampling — skip timesteps, 10–50 steps instead of 1000
- **Classifier-free guidance:** condition on text/class without a separate classifier — multiply noise predictions
- **Latent diffusion (Rombach et al. 2022):** run diffusion in compressed latent space (Stable Diffusion)
- **Score matching (Song & Ermon 2019):** equivalent training via $\nabla_x \log p(x)$ — the score function